In [1]:
from sklearn.datasets import make_classification
import torch

In [6]:
X, y = make_classification(
    n_samples = 10, 
    n_features = 2, 
    n_informative = 2,
    n_redundant = 0,
    n_classes = 2,
    random_state = 42
)

In [8]:
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

In [10]:
from torch.utils.data import Dataset, DataLoader

In [ ]:
class CustomDataset(Dataset): 

    def __init__(self, features, labels): 

        self.features = features
        self.labels = labels

    def __len__(self):
        return self.features.shape[0]

    def __getitem__(self, idx):

        return self.features[idx], self.labels[idx]


In [37]:
dataset = CustomDataset(X, y)

In [14]:
len(dataset)

10

In [15]:
dataset[0]

(tensor([ 1.0683, -0.9701]), tensor(1))

In [42]:
dataloader = DataLoader(dataset, batch_size=2, shuffle=True, num_workers=0, collate_fn=None, pin_memory=False, drop_last=False, timeout=0, worker_init_fn=None, sampler=None, batch_sampler=None)

In [43]:
for batch_features, batch_labels in dataloader: 
    print(batch_features)
    print(batch_labels)
    print("-"*50)

tensor([[ 1.7774,  1.5116],
        [-2.8954,  1.9769]])
tensor([1, 0])
--------------------------------------------------
tensor([[ 1.7273, -1.1858],
        [ 1.8997,  0.8344]])
tensor([1, 1])
--------------------------------------------------
tensor([[-0.5872, -1.9717],
        [-1.9629, -0.9923]])
tensor([0, 0])
--------------------------------------------------
tensor([[-0.9382, -0.5430],
        [-1.1402, -0.8388]])
tensor([1, 0])
--------------------------------------------------
tensor([[-0.7206, -0.9606],
        [ 1.0683, -0.9701]])
tensor([0, 1])
--------------------------------------------------


# Going back to BreastCancer


In [44]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [45]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [46]:
df.drop(columns=['id', 'Unnamed: 32'], inplace=True)

In [47]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['diagnosis']), df['diagnosis'], test_size=0.2, random_state=42)

In [48]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [49]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [50]:
X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.astype(np.float32))

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [53]:
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [63]:
import torch.nn as nn
class MySimpleNN(nn.Module): 

    def __init__(self, num_features): 
        super().__init__()
        self.linear = nn.Linear(in_features=num_features, out_features=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, features):
        out = self.linear(features)
        out = self.sigmoid(out)
        return out
    

In [64]:
learning_rate = 0.1
epochs = 25

In [65]:
model = MySimpleNN(num_features=X_train_tensor.shape[1])
loss_function = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [66]:
for epoch in range(epochs): 
    for batch_features, batch_labels in train_dataloader: 
        model.train()
        y_pred = model(batch_features).squeeze()
        loss = loss_function(y_pred, batch_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch: {epoch} | Loss: {loss.item():.4f}")

Epoch: 0 | Loss: 0.1572
Epoch: 1 | Loss: 0.2256
Epoch: 2 | Loss: 0.0837
Epoch: 3 | Loss: 0.0287
Epoch: 4 | Loss: 0.1320
Epoch: 5 | Loss: 0.0556
Epoch: 6 | Loss: 0.0373
Epoch: 7 | Loss: 0.0640
Epoch: 8 | Loss: 0.0983
Epoch: 9 | Loss: 0.0333
Epoch: 10 | Loss: 0.0292
Epoch: 11 | Loss: 0.0803
Epoch: 12 | Loss: 0.0554
Epoch: 13 | Loss: 0.0313
Epoch: 14 | Loss: 0.0168
Epoch: 15 | Loss: 0.0213
Epoch: 16 | Loss: 0.0632
Epoch: 17 | Loss: 0.0340
Epoch: 18 | Loss: 0.0603
Epoch: 19 | Loss: 0.0503
Epoch: 20 | Loss: 0.0642
Epoch: 21 | Loss: 0.0211
Epoch: 22 | Loss: 0.0165
Epoch: 23 | Loss: 0.0346
Epoch: 24 | Loss: 0.0701


In [67]:
model.eval() 
accuracy_list = []
with torch.no_grad():
    for batch_features, batch_labels in test_dataloader: 
        y_pred = model(batch_features).squeeze()
        y_pred_labels = (y_pred >= 0.5).float()
        accuracy = (y_pred_labels == batch_labels).float().mean()
        accuracy_list.append(accuracy.item())

overall_accuracy = sum(accuracy_list) / len(accuracy_list)
print(f"Test Accuracy: {overall_accuracy:.4f}")


Test Accuracy: 0.9922
